# Cookbook: Web IQ — Grounding Agents with Real-Time Web Search

Some questions can only be answered with information that changes by the hour: today's news, current prices, the latest release notes. **Web IQ** is Microsoft's agent-native grounding service — a standalone REST/MCP API that returns ranked, citation-ready context from the public web, news, images, and video, built for direct injection into an LLM's context window.

In this cookbook, you will:

- **Call the Web Search endpoint** and inspect grounding content and citations
- **Call the News Search endpoint** for the last 14 days of coverage
- **Call the Images Search endpoint**
- **Handle errors and rate limits** per the documented error schema
- **Configure the Web IQ MCP server** for use in VS Code / GitHub Copilot

## Prerequisites

You need:

1. **Web IQ access** — Web IQ is currently in limited access. Request access and generate an API key at the [Microsoft Web IQ Portal](https://webiq.microsoft.ai/profiles/).
2. **Python 3.10+**.

Create a `.env` file in this directory:

```
WEBIQ_API_KEY=<your-web-iq-api-key>
WEBIQ_BASE_URL=https://api.microsoft.ai/v3
```

> **💡 No Azure subscription needed.** Web IQ is a standalone, hosted service — there's no index, knowledge base, or Foundry project to provision for this cookbook.

## Setup

Install required packages and load environment variables.

In [ ]:
%%capture
%pip install -U requests python-dotenv

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

WEBIQ_API_KEY = os.environ["WEBIQ_API_KEY"]
WEBIQ_BASE_URL = os.environ.get("WEBIQ_BASE_URL", "https://api.microsoft.ai/v3")

HEADERS = {
    "x-apikey": WEBIQ_API_KEY,
    "content-type": "application/json",
}

print(f"Base URL: {WEBIQ_BASE_URL}")

---

## Step 1 — Web Search

`POST /search/web` returns ranked web pages with grounding content. Use `contentFormat="passage"` to get query-contextual extracted paragraphs instead of full-page HTML — this is the most token-efficient option for grounding an LLM.

In [ ]:
def webiq_post(path: str, payload: dict) -> dict:
    response = requests.post(f"{WEBIQ_BASE_URL}{path}", headers=HEADERS, json=payload, timeout=30)
    response.raise_for_status()
    return response.json()

web_results = webiq_post("/search/web", {
    "query": "latest trends in LLM RAG",
    "maxResults": 5,
    "language": "en",
    "region": "US",
    "contentFormat": "passage",
    "maxLength": 2000,
})

for result in web_results["webResults"]:
    print(f"📄 {result['title']}")
    print(f"   {result['url']}")
    print(f"   {result['content'][:200]}...\n")

print(f"Trace ID: {web_results['traceId']}")

---

## Step 2 — News Search

`POST /search/news` returns trusted news coverage from the **last 14 days only**. For older coverage, use Web Search instead.

In [ ]:
news_results = webiq_post("/search/news", {
    "query": "Microsoft AI announcements",
    "maxResults": 5,
    "language": "en",
    "region": "US",
})

for result in news_results["newsResults"]:
    print(f"📰 {result['title']} — {result.get('source', 'unknown source')}")
    print(f"   {result['url']}\n")

---

## Step 3 — Images Search

`POST /search/images` returns image results with captions, dimensions, and thumbnails — useful for multi-modal grounding.

In [ ]:
image_results = webiq_post("/search/images", {
    "query": "Microsoft IQ Series",
    "maxResults": 3,
    "language": "en",
    "region": "US",
})

for result in image_results["imageResults"]:
    print(f"🖼️ {result['title']}")
    print(f"   {result['url']} ({result.get('width')}x{result.get('height')})\n")

---

## Step 4 — Handle Errors and Rate Limits

Every Web IQ error response follows the same schema: `errorCode`, `errorCategory`, `userMessage`, and (for 429s and retryable 5xxs) `retryAfter`. Always check status codes rather than assuming success, and back off on `429`/`5xx`.

In [ ]:
import time


def webiq_post_with_retry(path: str, payload: dict, max_retries: int = 3) -> dict:
    for attempt in range(max_retries):
        response = requests.post(f"{WEBIQ_BASE_URL}{path}", headers=HEADERS, json=payload, timeout=30)
        if response.status_code == 200:
            return response.json()
        if response.status_code == 429 and attempt < max_retries - 1:
            error = response.json()
            wait_seconds = int(error.get("retryAfter", "5s").rstrip("s"))
            print(f"⏳ Rate limited (traceId={error.get('traceId')}). Waiting {wait_seconds}s before retry...")
            time.sleep(wait_seconds)
            continue
        response.raise_for_status()
    raise RuntimeError("Exceeded max retries")

# A deliberately invalid request to see the documented error schema (maxResults over the limit)
bad_response = requests.post(
    f"{WEBIQ_BASE_URL}/search/web",
    headers=HEADERS,
    json={"query": "test", "maxResults": 1000},
    timeout=30,
)
print(f"Status: {bad_response.status_code}")
print(json.dumps(bad_response.json(), indent=2))

---

## Step 5 — Configure the Web IQ MCP Server

Web IQ also exposes an MCP server over Streamable HTTP at `https://api.microsoft.ai/v3/mcp`, with `web`, `news`, `images`, `videos`, and `browse` tools. Add this to `.vscode/mcp.json` to use Web IQ directly from VS Code Copilot Chat:

```json
{
  "Search-MCP": {
    "url": "https://api.microsoft.ai/v3/mcp",
    "type": "http",
    "headers": {
      "x-apikey": "<your-api-key>"
    }
  }
}
```

Once configured, enable the `Search-MCP` tool in Copilot Chat's tools picker and ask questions like *"What's the latest news about the Microsoft IQ Series?"* — Copilot will call the `news` MCP tool automatically.

---

## Conclusion

In this cookbook we walked through grounding an application in the live public web using Web IQ:

1. **Web Search** — We retrieved ranked, citation-ready web pages with token-efficient `passage` content.
2. **News Search** — We retrieved trusted news coverage from the last 14 days.
3. **Images Search** — We retrieved image results for multi-modal grounding.
4. **Error handling** — We saw the standardized error schema and how to back off on `429`/`5xx` responses.
5. **MCP server** — We configured the Web IQ MCP endpoint for direct use in VS Code Copilot Chat.

### Where to go next

- **Try Browse** — Use `POST /browse` to extract content from a specific URL, with `liveCrawl="fallback"` for on-demand crawling of pages not yet indexed.
- **Try Videos Search** — Use `POST /search/videos` to retrieve video results with moments and embeddable players.
- **Try Auto (Beta)** — Use `POST /auto` for typed evidence with result-scoped citations across web, news, weather, and more in a single call.
- **Combine with Foundry IQ** — Ground the same agent in both your organizational knowledge (Foundry IQ) and the live web (Web IQ).
- **Explore the IQ Series** — Head back to the [Web IQ overview](../../README.md) or continue with [Foundry IQ](../../../Foundry-IQ/) and [Work IQ](../../../Work-IQ/).